# NB04 v3 — Diagnosis & Direct HDBSCAN (Tanpa Precomputed Matrix)

## Kenapa v1 & v2 Gagal?

| Versi | min_cluster_size | Hasil | Sebab |
|-------|-----------------|-------|-------|
| v1    | 10 – 50         | n_clusters **≥ 150** | fill=2.0 ciptakan ratusan mini-island |
| v2    | 50 – 500        | n_clusters **= 0**   | mcs besar → semua mini-island jadi noise |

**Root cause:** Pipeline precomputed matrix mengisi jarak non-k-NN dengan `2.0`
(max cosine distance). HDBSCAN melihat ribuan pulau terisolasi yang tidak bisa
digabungkan. Saat mcs kecil → ratusan pulau lolos. Saat mcs besar → semua jadi noise.

## Solusi

**Approach A (Primary):** Direct HDBSCAN langsung pada embedding ternormalisasi
- `metric='euclidean'` pada L2-normalized embedding ≡ cosine distance
- Tidak ada FAISS, tidak ada fill value, tidak ada masalah artifisial
- HDBSCAN melihat distribusi data sebenarnya

**Approach B (Comparison):** Precomputed matrix dengan fill diperbaiki
- fill = `max_actual_knn_dist + 0.05` (bukan 2.0 lagi) + k lebih besar (50, 100)

**Grid:** mcs=[30,50,75,100,150,200], ms=[5,10,15], csm=['eom','leaf'] → **36 kombinasi**


## 0. Install & Import

In [ ]:
import subprocess, sys

def install_faiss():
    for pkg in ["faiss-gpu-cu12", "faiss-gpu-cu11", "faiss-cpu"]:
        r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                           capture_output=True, text=True)
        if r.returncode == 0:
            print(f"✅ {pkg}"); return
    raise RuntimeError("Gagal install faiss")

try:
    import faiss; print(f"✅ faiss {faiss.__version__}")
except ImportError:
    install_faiss()

subprocess.run([sys.executable, "-m", "pip", "install",
                "hdbscan", "scikit-learn", "matplotlib", "pandas", "scipy", "-q"],
               check=True)


In [ ]:
import pickle, time, warnings
from pathlib import Path

import faiss, hdbscan
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

warnings.filterwarnings("ignore")
print(f"faiss {faiss.__version__}  |  hdbscan {hdbscan.__version__}")
try:
    ngpu = faiss.get_num_gpus()
    print(f"GPU: {ngpu} {'✅' if ngpu > 0 else '⚠️  CPU only'}")
except Exception:
    print("GPU: tidak tersedia")


## 1. Konfigurasi

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


In [ ]:
# ─── PATH ─────────────────────────────────────────────────────────────────────
INPUT_EMB_DIR = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings/output_nb01")
OUTPUT_DIR    = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings/output_nb04")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Approach A: Direct HDBSCAN — Parameter Grid ──────────────────────────────
# metric='euclidean' pada embedding ternormalisasi ≡ cosine distance
# Tidak ada fill value buatan, HDBSCAN melihat data apa adanya
MIN_CLUSTER_SIZE_GRID     = [30, 50, 75, 100, 150, 200]
MIN_SAMPLES_GRID          = [5, 10, 15]
CLUSTER_SELECTION_METHODS = ["eom", "leaf"]

TARGET_MAX_CLUSTERS = 150
MIN_COVERAGE        = 70.0

total_A = len(MIN_CLUSTER_SIZE_GRID) * len(MIN_SAMPLES_GRID) * len(CLUSTER_SELECTION_METHODS)

print("=== Approach A: Direct HDBSCAN ===")
print(f"  min_cluster_size         : {MIN_CLUSTER_SIZE_GRID}")
print(f"  min_samples              : {MIN_SAMPLES_GRID}")
print(f"  cluster_selection_method : {CLUSTER_SELECTION_METHODS}")
print(f"  Total kombinasi          : {total_A}")
print()
print(f"  Target: n_clusters < {TARGET_MAX_CLUSTERS},  coverage >= {MIN_COVERAGE}%")


## 2. Load & Normalize Embeddings

In [ ]:
embeddings = np.load(INPUT_EMB_DIR / "embeddings.npy").astype(np.float32)
N, D = embeddings.shape

# L2-normalize: euclidean distance pada normalized embedding = cosine distance
emb_norm = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

print(f"Embeddings : {embeddings.shape}  dtype={embeddings.dtype}")
print(f"Normalized : mean norm = {np.linalg.norm(emb_norm, axis=1).mean():.4f}  (harus 1.0)")
print()
print(f"Estimasi foto/orang  : {N/TARGET_MAX_CLUSTERS:.0f} (jika tepat {TARGET_MAX_CLUSTERS} orang)")
print(f"  → min_cluster_size ideal: {N/TARGET_MAX_CLUSTERS*0.3:.0f} – {N/TARGET_MAX_CLUSTERS:.0f}")


## 3. Diagnosis: Mengapa Fill=2.0 Bermasalah?

Cell ini menjalankan **3 konfigurasi diagnostik** untuk memperlihatkan masalah
pipeline precomputed sebelum dan sesudah perbaikan.


In [ ]:
# ─── Bangun diagnostik cepat ─────────────────────────────────────────────────
diag_k   = 25
diag_mcs = 100
diag_ms  = 10

faiss.normalize_L2(emb_f32 := embeddings.copy())
try:
    res = faiss.StandardGpuResources()
    idx = faiss.index_cpu_to_gpu(res, 0, faiss.IndexFlatIP(D))
except Exception:
    idx = faiss.IndexFlatIP(D)
idx.add(emb_f32)
sim, knn_idx = idx.search(emb_f32, diag_k + 1)
sim   = sim[:, 1:];  knn_idx = knn_idx[:, 1:]
dist  = np.clip(1.0 - sim, 0.0, 2.0).astype(np.float32)

rows  = np.repeat(np.arange(N), diag_k)
sparse_mat = csr_matrix((dist.ravel(), (rows, knn_idx.ravel())), shape=(N, N))
sparse_mat = sparse_mat.maximum(sparse_mat.T)

# Dense dengan fill=2.0 (cara lama)
dense_bad = np.full((N, N), 2.0, dtype=np.float64)
np.fill_diagonal(dense_bad, 0.0)
cx = sparse_mat.tocoo()
dense_bad[cx.row, cx.col] = cx.data.astype(np.float64)

# Dense dengan fill diperbaiki (cara baru)
fill_fixed = float(dist.max()) + 0.05
dense_fix  = np.full((N, N), fill_fixed, dtype=np.float64)
np.fill_diagonal(dense_fix, 0.0)
dense_fix[cx.row, cx.col] = cx.data.astype(np.float64)

print(f"Statistik jarak k-NN aktual (k={diag_k}):")
print(f"  min={dist.min():.4f}  mean={dist.mean():.4f}  "
      f"p95={np.percentile(dist,95):.4f}  max={dist.max():.4f}")
print(f"  fill lama : 2.0000  (×{2.0/dist.mean():.1f} rata-rata jarak aktual!)")
print(f"  fill baru : {fill_fixed:.4f}  (max_actual + 0.05)")
print()

for label, mat in [("fill=2.0 (lama)", dense_bad), (f"fill={fill_fixed:.2f} (baru)", dense_fix)]:
    t0 = time.time()
    cl = hdbscan.HDBSCAN(min_cluster_size=diag_mcs, min_samples=diag_ms,
                          metric="precomputed", cluster_selection_method="eom", n_jobs=-1)
    cl.fit(mat)
    lbl = cl.labels_
    n_cl = len(set(lbl[lbl >= 0]))
    cov  = (lbl >= 0).sum() / N * 100
    print(f"  Precomputed {label}:")
    print(f"    n_clusters={n_cl},  coverage={cov:.1f}%,  ({time.time()-t0:.0f}s)")

# Direct approach
t0 = time.time()
cl_direct = hdbscan.HDBSCAN(min_cluster_size=diag_mcs, min_samples=diag_ms,
                              metric="euclidean", cluster_selection_method="eom",
                              approx_min_span_tree=True, n_jobs=-1)
cl_direct.fit(emb_norm)
lbl = cl_direct.labels_
n_cl = len(set(lbl[lbl >= 0]))
cov  = (lbl >= 0).sum() / N * 100
print(f"  Direct HDBSCAN (euclidean, normalized):")
print(f"    n_clusters={n_cl},  coverage={cov:.1f}%,  ({time.time()-t0:.0f}s)")


## 4. Approach A: Grid Search — Direct HDBSCAN

HDBSCAN langsung pada embedding ternormalisasi.  
`metric='euclidean'` pada L2-normalized embedding ≡ cosine similarity.


In [ ]:
results_A = []
done, t_start = 0, time.time()

print(f"Approach A — Direct HDBSCAN: {total_A} kombinasi")
print("─" * 75)

for mcs in MIN_CLUSTER_SIZE_GRID:
    for ms in MIN_SAMPLES_GRID:
        for csm in CLUSTER_SELECTION_METHODS:
            t0    = time.time()
            done += 1
            try:
                clusterer = hdbscan.HDBSCAN(
                    min_cluster_size=mcs,
                    min_samples=ms,
                    metric="euclidean",
                    cluster_selection_method=csm,
                    approx_min_span_tree=True,
                    n_jobs=-1,
                )
                clusterer.fit(emb_norm)
                labels      = clusterer.labels_
                n_clusters  = int(len(set(labels[labels >= 0])))
                coverage    = float((labels >= 0).sum() / N * 100)
                noise_pct   = float((labels == -1).sum() / N * 100)
                dbcv_approx = float(clusterer.relative_validity_)
            except Exception as e:
                n_clusters, coverage, noise_pct, dbcv_approx = -1, -1.0, -1.0, float("nan")
                print(f"  ERROR: {e}")

            t_run = time.time() - t0
            results_A.append({
                "approach"                : "direct",
                "min_cluster_size"        : mcs,
                "min_samples"             : ms,
                "cluster_selection_method": csm,
                "n_clusters"              : n_clusters,
                "coverage_pct"            : round(coverage, 2),
                "noise_pct"               : round(noise_pct, 2),
                "dbcv_approx"             : round(dbcv_approx, 4),
                "runtime_s"               : round(t_run, 1),
            })

            t_el  = time.time() - t_start
            t_est = t_el / done * (total_A - done)
            flag  = " ✅" if n_clusters < TARGET_MAX_CLUSTERS and coverage >= MIN_COVERAGE else ""
            print(f"  [{done:2}/{total_A}] mcs={mcs:3} ms={ms:2} {csm:4} → "
                  f"n={n_clusters:4}, cov={coverage:5.1f}%, DBCV={dbcv_approx:+.3f}  "
                  f"{t_run:.0f}s  ~{t_est/60:.0f}mnt{flag}")

df_A = pd.DataFrame(results_A)
t_total = time.time() - t_start
print(f"\n✅ Selesai {t_total:.0f}s ({t_total/60:.1f} menit)")


## 5. Hasil & Analisis

In [ ]:
COLS = ["min_cluster_size","min_samples","cluster_selection_method",
        "n_clusters","coverage_pct","noise_pct","dbcv_approx"]

df_valid = df_A[df_A["n_clusters"] >= 0].copy()

print(f"Valid: {len(df_valid)}/{len(df_A)}")
if len(df_valid) > 0:
    print(f"n_clusters range : {df_valid.n_clusters.min()} – {df_valid.n_clusters.max()}")
    print(f"coverage range   : {df_valid.coverage_pct.min():.1f}% – {df_valid.coverage_pct.max():.1f}%")
    print(f"DBCV range       : {df_valid.dbcv_approx.min():.3f} – {df_valid.dbcv_approx.max():.3f}")
    print()
    print("=== 10 hasil n_clusters TERKECIL ===")
    print(df_valid.sort_values("n_clusters").head(10)[COLS].to_string(index=False))
    print()

    df_target = df_valid[
        (df_valid.n_clusters < TARGET_MAX_CLUSTERS) &
        (df_valid.coverage_pct >= MIN_COVERAGE)
    ].sort_values("dbcv_approx", ascending=False)

    print(f"Memenuhi target (n<{TARGET_MAX_CLUSTERS}, cov≥{MIN_COVERAGE}%) : {len(df_target)}")
    if len(df_target) > 0:
        print()
        print("=== TOP 20 ===")
        print(df_target[COLS].head(20).to_string(index=False))
else:
    print("⚠️  Semua kombinasi error atau tidak valid.")
    print("Pastikan embeddings.npy terload dengan benar.")


In [ ]:
if len(df_valid) == 0:
    print("Tidak ada data untuk divisualisasi.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    colors = {"eom": "#2196F3", "leaf": "#FF5722"}
    markers = {"eom": "o", "leaf": "s"}

    ms_show = MIN_SAMPLES_GRID[1]  # min_samples tengah

    for csm in CLUSTER_SELECTION_METHODS:
        df_c = df_valid[(df_valid.cluster_selection_method == csm) &
                        (df_valid.min_samples == ms_show)
                       ].sort_values("min_cluster_size")
        for ax, (metric, ylabel, hval) in zip(axes, [
            ("n_clusters",  "Jumlah Cluster",   TARGET_MAX_CLUSTERS),
            ("coverage_pct","Coverage (%)",      MIN_COVERAGE),
            ("dbcv_approx", "DBCV (approx)",     None),
        ]):
            ax.plot(df_c.min_cluster_size, df_c[metric],
                    marker=markers[csm], label=f"csm={csm}",
                    color=colors[csm], linewidth=2, markersize=7)

    for ax, (metric, ylabel, hval) in zip(axes, [
        ("n_clusters",  "Jumlah Cluster",   TARGET_MAX_CLUSTERS),
        ("coverage_pct","Coverage (%)",      MIN_COVERAGE),
        ("dbcv_approx", "DBCV (approx)",     None),
    ]):
        if hval is not None:
            ax.axhline(hval, color="red", linestyle="--", linewidth=1.5,
                       label=f"Target ({hval})")
        ax.set_xlabel("min_cluster_size", fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(f"{ylabel} vs min_cluster_size\n(ms={ms_show}, direct HDBSCAN)",
                     fontsize=10, fontweight="bold")
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "lineplot_v3.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Tersimpan: lineplot_v3.png")


In [ ]:
# ── Heatmap n_clusters per csm ───────────────────────────────────────────────
if len(df_valid) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("n_clusters (Direct HDBSCAN) — bold = memenuhi target",
                 fontsize=12, fontweight="bold")

    for ax, csm in zip(axes, CLUSTER_SELECTION_METHODS):
        df_c = df_valid[df_valid.cluster_selection_method == csm]
        pivot = df_c.pivot(index="min_cluster_size", columns="min_samples",
                           values="n_clusters")
        if pivot.empty:
            continue
        im = ax.imshow(pivot.values, cmap="RdYlGn_r", aspect="auto")
        plt.colorbar(im, ax=ax, shrink=0.85)
        ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
        ax.set_yticks(range(len(pivot.index)));   ax.set_yticklabels(pivot.index)
        ax.set_xlabel("min_samples"); ax.set_ylabel("min_cluster_size")
        ax.set_title(f"csm='{csm}'", fontsize=11, fontweight="bold")

        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                val = pivot.values[i, j]
                if not np.isnan(val):
                    fw = "bold" if val < TARGET_MAX_CLUSTERS else "normal"
                    ax.text(j, i, f"{val:.0f}", ha="center", va="center",
                            fontsize=9, fontweight=fw)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "heatmap_v3.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Tersimpan: heatmap_v3.png")


## 6. Parameter Terbaik

In [ ]:
print("=" * 65)
print("  PARAMETER TERBAIK — NB04 v3 (Direct HDBSCAN)")
print("=" * 65)

best_params = {}

if len(df_valid) == 0:
    print("⚠️  Tidak ada hasil valid. Cek embeddings.npy.")
else:
    df_best = df_valid[
        (df_valid.n_clusters < TARGET_MAX_CLUSTERS) &
        (df_valid.coverage_pct >= MIN_COVERAGE)
    ].sort_values("dbcv_approx", ascending=False)

    if len(df_best) > 0:
        best = df_best.iloc[0]
        print(f"  min_cluster_size         : {int(best.min_cluster_size)}")
        print(f"  min_samples              : {int(best.min_samples)}")
        print(f"  cluster_selection_method : {best.cluster_selection_method}")
        print()
        print(f"  n_clusters  : {int(best.n_clusters)}")
        print(f"  coverage    : {best.coverage_pct:.2f}%")
        print(f"  noise       : {best.noise_pct:.2f}%")
        print(f"  DBCV approx : {best.dbcv_approx:.4f}")
        print("=" * 65)
        print()
        print("Gunakan di NB05 (metric='euclidean', embed langsung):")
        print(f"  MIN_CLUSTER_SIZE          = {int(best.min_cluster_size)}")
        print(f"  MIN_SAMPLES               = {int(best.min_samples)}")
        print(f"  CLUSTER_SELECTION_METHOD  = \"{best.cluster_selection_method}\"")
        best_params = dict(best)
    else:
        print(f"⚠️  Tidak ada yang memenuhi n<{TARGET_MAX_CLUSTERS} + cov≥{MIN_COVERAGE}%")
        print()
        print("Range n_clusters yang tersedia:")
        print(df_valid.groupby("cluster_selection_method")["n_clusters"].describe().round(1))
        print()
        print("Top 5 coverage tertinggi (tanpa filter n_clusters):")
        print(df_valid.sort_values("coverage_pct", ascending=False)[COLS].head(5).to_string(index=False))
        print()
        print("→ Coba turunkan TARGET_MAX_CLUSTERS ke nilai n_clusters minimum yang ditemukan.")


## 7. Simpan Hasil

In [ ]:
csv_path = OUTPUT_DIR / "tuning_results_v3.csv"
pkl_path = OUTPUT_DIR / "tuning_results_v3.pkl"

df_A.to_csv(csv_path, index=False)
with open(pkl_path, "wb") as f:
    pickle.dump({"results_df": df_A, "best_params": best_params}, f)

print(f"CSV : {csv_path}  ({csv_path.stat().st_size/1024:.1f} KB)")
print(f"PKL : {pkl_path}  ({pkl_path.stat().st_size/1024:.1f} KB)")
print(f"\nTotal: {len(df_A)} | Valid: {len(df_valid)} | "
      f"Memenuhi target: {len(df_best) if 'df_best' in dir() else '?'}")
